**Q1. Word Count - Count the frequency of each word
Input:
• hadoop is fast
• hadoop is scalable**

Manual Aproach

In [ ]:
# Mapper
data = ["hadoop is fast", "hadoop is scalable"]

mapped = []

for line in data:
    words = line.split()
    for word in words:
        mapped.append((word, 1))

mapped

[('hadoop', 1),
 ('is', 1),
 ('fast', 1),
 ('hadoop', 1),
 ('is', 1),
 ('scalable', 1)]

In [ ]:
# Shuffle
shuffle = {}

for word, count in mapped:
    if word not in shuffle:
        shuffle[word] = []
    shuffle[word].append(count)

shuffle

{'hadoop': [1, 1], 'is': [1, 1], 'fast': [1], 'scalable': [1]}

In [ ]:
# Reducer
reduced = {}

for word in shuffle:
    reduced[word] = sum(shuffle[word])

reduced

{'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}

mrjob framework

In [ ]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 4.6 MB/s eta 0:00:00


In [ ]:
%%writefile wordcount.py
from mrjob.job import MRJob

class WordCount(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Writing wordcount.py


In [ ]:
%%writefile input.txt
hadoop is fast
hadoop is scalable

Writing input.txt


In [ ]:
!python wordcount.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260427.172431.401834
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260427.172431.401834/output
Streaming final output from /tmp/wordcount.root.20260427.172431.401834/output...
"is"	2
"scalable"	1
"fast"	1
"hadoop"	2
Removing temp directory /tmp/wordcount.root.20260427.172431.401834...


**Q2. Character Count - Count the frequency of each character (ignore spaces).
Input: big data**

Manual Aproach

In [ ]:
data = "big data"
mapped = []
for char in data:
    if char != " ":
        mapped.append((char, 1))
print("Mapper Output:", mapped)
shuffle = {}
for char, count in mapped:
    if char not in shuffle:
        shuffle[char] = []
    shuffle[char].append(count)

print("Shuffle Output:", shuffle)
reduced = {}
for char, counts in shuffle.items():
    reduced[char] = sum(counts)
print("Final Output:", reduced)

Mapper Output: [('b', 1), ('i', 1), ('g', 1), ('d', 1), ('a', 1), ('t', 1), ('a', 1)]
Shuffle Output: {'b': [1], 'i': [1], 'g': [1], 'd': [1], 'a': [1, 1], 't': [1]}
Final Output: {'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


mrjob framework

In [ ]:
%%writefile charcount.py
from mrjob.job import MRJob
class CharCount(MRJob):
    def mapper(self, _, line):
        for char in line:
            if char != " ":
                yield char, 1
    def reducer(self, key, values):
        yield key, sum(values)
if __name__ == "__main__":
    CharCount.run()

Writing charcount.py


In [ ]:
%%writefile input.txt
big data

Overwriting input.txt


In [ ]:
!python charcount.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charcount.root.20260427.172920.803478
Running step 1 of 1...
job output is in /tmp/charcount.root.20260427.172920.803478/output
Streaming final output from /tmp/charcount.root.20260427.172920.803478/output...
"b"	1
"d"	1
"g"	1
"i"	1
"t"	1
"a"	2
Removing temp directory /tmp/charcount.root.20260427.172920.803478...


**Q3. Average Word Length (Per Word) - Compute the average length of each word.
Input: data science data big data**

Manual Aproach

In [ ]:
data = "data science data big data".split()
mapped = []
for word in data:
    mapped.append((word, (len(word), 1)))
    # (word, (length, count))
print("Mapper Output:", mapped)
shuffle = {}
for word, value in mapped:
    if word not in shuffle:
        shuffle[word] = []
    shuffle[word].append(value)
print("Shuffle Output:", shuffle)
reduced = {}
for word, values in shuffle.items():
    total_length = sum(v[0] for v in values)
    total_count = sum(v[1] for v in values)
    reduced[word] = total_length / total_count
print("Final Output:", reduced)

Mapper Output: [('data', (4, 1)), ('science', (7, 1)), ('data', (4, 1)), ('big', (3, 1)), ('data', (4, 1))]
Shuffle Output: {'data': [(4, 1), (4, 1), (4, 1)], 'science': [(7, 1)], 'big': [(3, 1)]}
Final Output: {'data': 4.0, 'science': 7.0, 'big': 3.0}


mrjob framework

In [ ]:
%%writefile avg_word_length.py
from mrjob.job import MRJob
class AvgWordLength(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word, (len(word), 1)
    def reducer(self, key, values):
        total_length = 0
        total_count = 0
        for length, count in values:
            total_length += length
            total_count += count
        yield key, total_length / total_count
if __name__ == "__main__":
    AvgWordLength.run()

Writing avg_word_length.py


In [ ]:
%%writefile input.txt
data science data big data

Overwriting input.txt


In [ ]:
!python avg_word_length.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_word_length.root.20260427.173059.144892
Running step 1 of 1...
job output is in /tmp/avg_word_length.root.20260427.173059.144892/output
Streaming final output from /tmp/avg_word_length.root.20260427.173059.144892/output...
"science"	7.0
"big"	3.0
"data"	4.0
Removing temp directory /tmp/avg_word_length.root.20260427.173059.144892...


**Q4.  Global Average Word Length - Compute the average length of all words.
Input: hadoop mapreduce spark**

Manual Aproach

In [ ]:
data = "hadoop mapreduce spark".split()
mapped = []
for word in data:
    mapped.append((1, (len(word), 1)))
print("Mapper Output:", mapped)
shuffle = {}
for key, value in mapped:
    if key not in shuffle:
        shuffle[key] = []
    shuffle[key].append(value)
print("Shuffle Output:", shuffle)
for key, values in shuffle.items():
    total_length = sum(v[0] for v in values)
    total_count = sum(v[1] for v in values)
    avg = total_length / total_count
print("Global Average:", avg)

Mapper Output: [(1, (6, 1)), (1, (9, 1)), (1, (5, 1))]
Shuffle Output: {1: [(6, 1), (9, 1), (5, 1)]}
Global Average: 6.666666666666667


mrjob framework

In [ ]:
%%writefile global_avg.py
from mrjob.job import MRJob
class GlobalAvg(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield "global", (len(word), 1)
    def reducer(self, key, values):
        total_length = 0
        total_count = 0
        for length, count in values:
            total_length += length
            total_count += count
        yield key, total_length / total_count
if __name__ == "__main__":
    GlobalAvg.run()

Writing global_avg.py


In [ ]:
%%writefile input.txt
hadoop mapreduce spark

Overwriting input.txt


In [ ]:
!python global_avg.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/global_avg.root.20260427.173459.220873
Running step 1 of 1...
job output is in /tmp/global_avg.root.20260427.173459.220873/output
Streaming final output from /tmp/global_avg.root.20260427.173459.220873/output...
"global"	6.666666666666667
Removing temp directory /tmp/global_avg.root.20260427.173459.220873...


**Q5. Perform Q1-Q4 on the file available on the link:**
https://drive.google.com/file/d/16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas/view?usp=sharing

**Also find Top 5 most frequent words**

In [ ]:
# Download file
!gdown https://drive.google.com/uc?id=16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas

# Read file
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.readlines()

Downloading...
From: https://drive.google.com/uc?id=16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas
To: /content/shakespeare.txt
100% 5.59M/5.59M [00:00<00:00, 40.3MB/s]


Manual

In [ ]:
#Word Count
mapped = []
for line in data:
    for word in line.split():
        mapped.append((word.lower(), 1))

shuffle = {}
for word, count in mapped:
    if word not in shuffle:
        shuffle[word] = []
    shuffle[word].append(count)

word_count = {word: sum(shuffle[word]) for word in shuffle}
print("Word Count:", list(word_count.items())[:10])

#Character Count
char_map = []
for line in data:
    for ch in line:
        if ch != " " and ch != "\n":
            char_map.append((ch, 1))

char_shuffle = {}
for ch, count in char_map:
    if ch not in char_shuffle:
        char_shuffle[ch] = []
    char_shuffle[ch].append(count)

char_count = {ch: sum(char_shuffle[ch]) for ch in char_shuffle}
print("Character Count:", list(char_count.items())[:10])

#Avg Word Length per Word
from collections import defaultdict

word_len = defaultdict(list)
for line in data:
    for word in line.split():
        word_len[word].append(len(word))

avg_word_len = {w: sum(word_len[w])/len(word_len[w]) for w in word_len}
print("Avg Word Length per Word:", list(avg_word_len.items())[:10])

#Global Avg Word Length
all_words = []
for line in data:
    all_words.extend(line.split())

total_len = sum(len(w) for w in all_words)
global_avg = total_len / len(all_words)
print("Global Avg Word Length:", global_avg)

#Top 5 Words
from collections import Counter

top5 = Counter(all_words).most_common(5)
print("Top 5 Words:", top5)

Word Count: [('\ufeffthe', 1), ('project', 320), ('gutenberg', 250), ('ebook', 13), ('of', 18126), ('the', 27729), ('complete', 243), ('works', 268), ('william', 311), ('shakespeare,', 2)]
Character Count: [('\ufeff', 1), ('T', 39878), ('h', 218875), ('e', 406157), ('P', 12038), ('r', 209907), ('o', 282560), ('j', 2788), ('c', 67194), ('t', 291243)]
Avg Word Length per Word: [('\ufeffThe', 4.0), ('Project', 7.0), ('Gutenberg', 9.0), ('EBook', 5.0), ('of', 2.0), ('The', 3.0), ('Complete', 8.0), ('Works', 5.0), ('William', 7.0), ('Shakespeare,', 12.0)]
Global Avg Word Length: 4.484685214825106
Top 5 Words: [('the', 23407), ('I', 19540), ('and', 18358), ('to', 15682), ('of', 15649)]


mrjob framework

In [ ]:
# Install
!pip install mrjob gdown

# Download file
!gdown https://drive.google.com/uc?id=16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas

# Create .py file (NO %%writefile)
code = '''
from mrjob.job import MRJob
from mrjob.step import MRStep

class Top5Words(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_count),
            MRStep(reducer=self.reducer_top5)
        ]

    def mapper(self, _, line):
        for word in line.split():
            yield word.lower(), 1

    def reducer_count(self, key, values):
        yield None, (sum(values), key)

    def reducer_top5(self, _, values):
        top5 = sorted(values, reverse=True)[:5]
        for count, word in top5:
            yield word, count

if __name__ == "__main__":
    Top5Words.run()
'''

with open("top5_words.py", "w") as f:
    f.write(code)

# Run
!python top5_words.py shakespeare.txt

Downloading...
From: https://drive.google.com/uc?id=16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas
To: /content/shakespeare.txt
100% 5.59M/5.59M [00:00<00:00, 42.0MB/s]
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/top5_words.root.20260427.182008.516784
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/top5_words.root.20260427.182008.516784/output
Streaming final output from /tmp/top5_words.root.20260427.182008.516784/output...
"the"	27729
"and"	26099
"i"	19540
"to"	18762
"of"	18126
Removing temp directory /tmp/top5_words.root.20260427.182008.516784...


**Q6.  Compute average marks for each student.**

Input:

A 80

B 70

A 90

B 60

A 100

Manual Aproach

In [ ]:
data = ["A 80", "B 70", "A 90", "B 60", "A 100"]
mapped = []
for line in data:
    student, marks = line.split()
    mapped.append((student, (int(marks), 1)))
print("Mapper Output:", mapped)
shuffle = {}
for student, value in mapped:
    if student not in shuffle:
        shuffle[student] = []
    shuffle[student].append(value)
print("Shuffle Output:", shuffle)
reduced = {}
for student, values in shuffle.items():
    total_marks = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    reduced[student] = total_marks / count
print("Final Output:", reduced)

Mapper Output: [('A', (80, 1)), ('B', (70, 1)), ('A', (90, 1)), ('B', (60, 1)), ('A', (100, 1))]
Shuffle Output: {'A': [(80, 1), (90, 1), (100, 1)], 'B': [(70, 1), (60, 1)]}
Final Output: {'A': 90.0, 'B': 65.0}


mrjob framework

In [ ]:
%%writefile avg_marks.py
from mrjob.job import MRJob
class AvgMarks(MRJob):
    def mapper(self, _, line):
        student, marks = line.split()
        yield student, (int(marks), 1)
    def reducer(self, key, values):
        total_marks = 0
        count = 0
        for marks, c in values:
            total_marks += marks
            count += c
        yield key, total_marks / count
if __name__ == "__main__":
    AvgMarks.run()

Writing avg_marks.py


In [ ]:
%%writefile marks.txt
A 80
B 70
A 90
B 60
A 100

Writing marks.txt


In [ ]:
!python avg_marks.py marks.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_marks.root.20260427.182356.554725
Running step 1 of 1...
job output is in /tmp/avg_marks.root.20260427.182356.554725/output
Streaming final output from /tmp/avg_marks.root.20260427.182356.554725/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/avg_marks.root.20260427.182356.554725...


**Q7.  Compute average salary per department and Highest Paid Department (Based on Average
Salary)**

Input:

HR 50000

IT 70000

HR 60000

IT 80000

Manual Aproach

In [ ]:
data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]
mapped = []
for line in data:
    dept, salary = line.split()
    mapped.append((dept, (int(salary), 1)))
print("Mapper Output:", mapped)
shuffle = {}
for dept, value in mapped:
    if dept not in shuffle:
        shuffle[dept] = []
    shuffle[dept].append(value)
print("Shuffle Output:", shuffle)
avg_salary = {}
for dept, values in shuffle.items():
    total_salary = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    avg_salary[dept] = total_salary / count
print("Average Salary:", avg_salary)
highest_dept = max(avg_salary, key=avg_salary.get)
print("Highest Paid Department:", highest_dept, avg_salary[highest_dept])

Mapper Output: [('HR', (50000, 1)), ('IT', (70000, 1)), ('HR', (60000, 1)), ('IT', (80000, 1))]
Shuffle Output: {'HR': [(50000, 1), (60000, 1)], 'IT': [(70000, 1), (80000, 1)]}
Average Salary: {'HR': 55000.0, 'IT': 75000.0}
Highest Paid Department: IT 75000.0


mrjob framework

In [ ]:
%%writefile dept_salary.py
from mrjob.job import MRJob
from mrjob.step import MRStep
class DeptSalary(MRJob):
    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_avg),
            MRStep(reducer=self.reducer_max)
        ]
    def mapper(self, _, line):
        dept, salary = line.split()
        yield dept, (int(salary), 1)
    def reducer_avg(self, key, values):
        total_salary = 0
        count = 0
        for salary, c in values:
            total_salary += salary
            count += c
        avg = total_salary / count
        yield None, (avg, key)
    def reducer_max(self, _, values):
        max_dept = max(values)   # compares by avg
        yield max_dept[1], max_dept[0]
if __name__ == "__main__":
    DeptSalary.run()

Writing dept_salary.py


In [ ]:
%%writefile salary.txt
HR 50000
IT 70000
HR 60000
IT 80000

Writing salary.txt


In [ ]:
!python dept_salary.py salary.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/dept_salary.root.20260427.182612.582243
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/dept_salary.root.20260427.182612.582243/output
Streaming final output from /tmp/dept_salary.root.20260427.182612.582243/output...
"IT"	75000.0
Removing temp directory /tmp/dept_salary.root.20260427.182612.582243...


**Q8. Computer average temperature per country**

New York,38

London,29

Tokyo,35

New York,32

Delhi,45

Ambala,35

Manual Aproach

In [ ]:
city_to_country = {
    "New York": "USA",
    "London": "UK",
    "Tokyo": "Japan",
    "Delhi": "India",
    "Ambala": "India"
}

In [ ]:
data = [
    "New York,38",
    "London,29",
    "Tokyo,35",
    "New York,32",
    "Delhi,45",
    "Ambala,35"
]
mapped = []
for line in data:
    city, temp = line.split(",")
    country = city_to_country[city]
    mapped.append((country, (int(temp), 1)))
print("Mapper Output:", mapped)
shuffle = {}
for country, value in mapped:
    if country not in shuffle:
        shuffle[country] = []
    shuffle[country].append(value)
print("Shuffle Output:", shuffle)
avg_temp = {}
for country, values in shuffle.items():
    total_temp = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    avg_temp[country] = total_temp / count
print("Average Temperature per Country:", avg_temp)

Mapper Output: [('USA', (38, 1)), ('UK', (29, 1)), ('Japan', (35, 1)), ('USA', (32, 1)), ('India', (45, 1)), ('India', (35, 1))]
Shuffle Output: {'USA': [(38, 1), (32, 1)], 'UK': [(29, 1)], 'Japan': [(35, 1)], 'India': [(45, 1), (35, 1)]}
Average Temperature per Country: {'USA': 35.0, 'UK': 29.0, 'Japan': 35.0, 'India': 40.0}


mrjob framework

In [ ]:
%%writefile avg_temp.py
from mrjob.job import MRJob
class AvgTemp(MRJob):
    def mapper(self, _, line):
        city, temp = line.split(",")
        city_to_country = {
            "New York": "USA",
            "London": "UK",
            "Tokyo": "Japan",
            "Delhi": "India",
            "Ambala": "India"
        }
        country = city_to_country[city]
        yield country, (int(temp), 1)
    def reducer(self, key, values):
        total_temp = 0
        count = 0
        for temp, c in values:
            total_temp += temp
            count += c
        yield key, total_temp / count
if __name__ == "__main__":
    AvgTemp.run()

Writing avg_temp.py


In [ ]:
%%writefile temp.txt
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

Writing temp.txt


In [ ]:
!python avg_temp.py temp.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_temp.root.20260427.182743.105492
Running step 1 of 1...
job output is in /tmp/avg_temp.root.20260427.182743.105492/output
Streaming final output from /tmp/avg_temp.root.20260427.182743.105492/output...
"Japan"	35.0
"UK"	29.0
"USA"	35.0
"India"	40.0
Removing temp directory /tmp/avg_temp.root.20260427.182743.105492...


**Q9. Redo Q8 with the dataset available on:**

https://www.kaggle.com/datasets/heemalichaudhari/global-land-temperatures

Manual

In [ ]:
import pandas as pd
df = pd.read_csv("/content/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv")
df = df.dropna(subset=["AverageTemperature", "Country"])
mapped = []
for _, row in df.iterrows():
    mapped.append((row["Country"], (row["AverageTemperature"], 1)))
shuffle = {}
for country, value in mapped:
    if country not in shuffle:
        shuffle[country] = []
    shuffle[country].append(value)
avg_temp = {}
for country, values in shuffle.items():
    total_temp = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    avg_temp[country] = total_temp / count
print(dict(list(avg_temp.items())[:10]))

{"Côte D'Ivoire": 26.16373719752392, 'Ethiopia': 17.52507266229899, 'India': 25.80930923845554, 'Syria': 17.37058733360226, 'Egypt': 20.900406225165565, 'Turkey': 13.790997727026735, 'Iraq': 22.61434568965517, 'Thailand': 27.164733303650937, 'Brazil': 22.847555235192356, 'Germany': 8.916233733417561}


mrjob framework

In [ ]:
%%writefile avg_temp_country.py
from mrjob.job import MRJob
import csv

class AvgTempCountry(MRJob):

    def mapper(self, _, line):
        try:
            row = next(csv.reader([line]))
            if row[1] == "AverageTemperature":
                return  # skip header

            temp = row[1]
            country = row[3]

            if temp:
                yield country, (float(temp), 1)
        except:
            pass

    def reducer(self, key, values):
        total = 0
        count = 0
        for temp, c in values:
            total += temp
            count += c
        yield key, total / count

if __name__ == "__main__":
    AvgTempCountry.run()

Writing avg_temp_country.py


In [ ]:
!python avg_temp_country.py /content/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_temp_country.root.20260427.184151.519787
Running step 1 of 1...
job output is in /tmp/avg_temp_country.root.20260427.184151.519787/output
Streaming final output from /tmp/avg_temp_country.root.20260427.184151.519787/output...
"Dalian"	10.229076814988273
"Dar Es Salaam"	25.74467150063052
"Delhi"	25.1658609022556
"Dhaka"	25.490567930489743
"Durban"	20.351292044847874
"Faisalabad"	24.138396543446955
"Fortaleza"	27.008639541892737
"Gizeh"	21.221259213759268
"Guangzhou"	21.60868426103649
"Harare"	20.207842011834327
"Harbin"	3.625744065602067
"Ho Chi Minh City"	27.193983566940577
"Hyderabad"	26.869334928229648
"Ibadan"	26.373105171411947
"Istanbul"	13.50740903348075
"Izmir"	17.278064578005115
"Jaipur"	25.39305794080872
"Jakarta"	26.508209686003163
"Jiddah"	27.692066017316
"Jinan"	13.089813819577751
"Kabul"	14.342918906176331
"Kano"	26.140536897152828
"Kanpur"	24.760040